<a href="https://colab.research.google.com/github/MizushimaToshihiko/Kaggle/blob/main/ps-s6e5-Predicting-F1-Pit-Stops/experiments/exp_20260521_047_lgb_shared001v2_features_optuna_search_min/PS_S6E5_047_LGB_Shared001v2_Features_Optuna_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


### Kaggle認証情報の自動設定
Colabの左側にある「🔑(シークレット)」に `KAGGLE_USERNAME` と `KAGGLE_KEY` を登録しておくと、以下のコードで自動的に認証が完了します。

In [31]:
import os
from google.colab import userdata

# シークレットから情報を取得（未設定の場合はエラーになります）
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
    print("Kaggle credentials set successfully.")
except:
    print("Kaggle credentials not found in Colab Secrets.")

Kaggle credentials set successfully.


これで、ノートブック内の `kagglehub.dataset_download()` などの関数が、ユーザー対話なしでファイルをダウンロード・参照できるようになります。ダウンロード先はColab環境内のローカルディスクなので、ランタイムが切れると消去されますが、Google Driveを介さないため非常に高速です。

In [32]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e5_path = kagglehub.competition_download('playground-series-s6e5')
aadigupta1601_f1_strategy_dataset_pit_stop_prediction_path = kagglehub.dataset_download('aadigupta1601/f1-strategy-dataset-pit-stop-prediction')
woodshole_playground_series_s6_e5_weather_conditions_path = kagglehub.dataset_download('woodshole/playground-series-s6-e5-weather-conditions')

print('Data source import complete.')


Using Colab cache for faster access to the 'f1-strategy-dataset-pit-stop-prediction' dataset.
Using Colab cache for faster access to the 'playground-series-s6-e5-weather-conditions' dataset.
Data source import complete.


# exp_20260521_047_lgb_shared001v2_features_optuna_search_min

Base: exp_20260519_040_lgb_shared001v2_features_min.

Purpose: keep the 040 LGB shared001v2 feature set fixed and run an Optuna search over LightGBM hyperparameters only.

This is stage 1 of 2. It does not create formal OOF/pred/submission for blend. If the search is useful, apply `best_params_047.json` in 048 refit.

Guardrails: no new feature engineering, no 039 LOG features, no Normalized_TyreLife/proxy, no pseudo-label, no endpoint/future proxy.


In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
# ============================================================
# PS S6E5 - exp_20260521_047_lgb_shared001v2_features_optuna_search_min
#
# Base:
#   040: exp_20260519_040_lgb_shared001v2_features_min
#
# Purpose:
#   040 LGB shared001v2 features構成を固定し、LightGBMの主要ハイパラだけをOptunaで探索する。
#
# Fixed from 040:
#   - LGB shared001v2-style feature representation
#   - weather aggregate / year-stint flags / progress-window features
#   - fold-safe target encoding and count/frequency features
#   - original rows appended to fold train side only
#   - target x Year stratification
#
# Guardrails:
#   - no new feature engineering
#   - no 039 LOG features
#   - no Normalized_TyreLife/proxy
#   - no pseudo-label
#   - no future endpoint proxy
# ============================================================

In [35]:
import os
import re
import gc
import json
import random
import warnings
import pickle
from pathlib import Path
from difflib import SequenceMatcher
import sys
import subprocess

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, TargetEncoder

import lightgbm as lgb
try:
  import optuna
except Exception:
    print("optuna not found. Installing optuna...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [36]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260521_047_lgb_shared001v2_features_optuna_search_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    # Use paths from kagglehub downloads
    COMP_BASE = Path(playground_series_s6e5_path)
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    # Use path from kagglehub for the original dataset
    ORIG_PATH = Path(aadigupta1601_f1_strategy_dataset_pit_stop_prediction_path) / "f1_strategy_dataset_v4.csv"

    # Use path from kagglehub for weather conditions
    WEATHER_CANDIDATE_PATHS = [
        Path(woodshole_playground_series_s6_e5_weather_conditions_path) / "F1_Weather_2022_2025.csv",
    ]

    OUTDIR = Path(f"/content/drive/MyDrive/Kaggle/PS S6E5 Predicting F1 Pit Stops/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"
    DANGER_COL = "Normalized_TyreLife"

    SEED = 3407
    SEEDS_BASE = [3407, 42]
    N_SPLITS = 5

    USE_ORIGINAL_ROWS = True
    USE_TARGET_ENCODING = True

    # Model Hyperparameters
    LGB_N_ESTIMATORS = 9000
    LGB_LEARNING_RATE = 0.022
    LGB_NUM_LEAVES = 31
    LGB_MIN_CHILD_SAMPLES = 120
    LGB_SUBSAMPLE = 0.88
    LGB_SUBSAMPLE_FREQ = 1
    LGB_COLSAMPLE_BYTREE = 0.82
    LGB_REG_ALPHA = 0.25
    LGB_REG_LAMBDA = 10.0
    LGB_MAX_BIN = 255
    LGB_EARLY_STOPPING = 450

    LGB_DEVICE = "gpu"

    USE_PROGRESS_WINDOW_TE = True
    PROGRESS_BINS = 20

    PROGRESS_WINDOW_CAT_COLS = [
        "RaceProgressBin_20",
        "Race_RaceProgressBin_20",
        "Race_Compound_RaceProgressBin_20",
    ]

    PROGRESS_WINDOW_TE_COLS = [
        "Race_RaceProgressBin_20",
        "Race_Compound_RaceProgressBin_20",
    ]

    USE_YEAR_STINT_FLAGS_025 = True
    ADD_YEAR_STINT_CROSSES_025 = True
    ADD_YEAR_STINT_TE_025 = False

    USE_SHARED001V2_FEATURES_040 = True
    KEEP_RAW_PITSTOP_040 = True
    ADD_RACE_COMPOUND_TE_040 = True

    OPTUNA_N_TRIALS = 40
    OPTUNA_TIMEOUT_SEC = 60 * 60 * 10
    OPTUNA_SEEDS = [3407]
    BASELINE_040_CV = 0.9524028165133429
    BASELINE_040_PUBLIC_LB = 0.95195

CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [37]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in out.columns:
        dtype = out[col].dtype

        if pd.api.types.is_integer_dtype(dtype):
            c_min = out[col].min()
            c_max = out[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")

    return out


def safe_divide(a, b, eps=1e-6):
    return a / (b + eps)


def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def find_weather_path() -> Path:
    for p in CFG.WEATHER_CANDIDATE_PATHS:
        if p.exists():
            return p
    raise FileNotFoundError(
        "F1_Weather_2022_2025.csv not found. Add Kaggle dataset: "
        "woodshole/playground-series-s6-e5-weather-conditions"
    )


seed_everything(CFG.SEED)




In [38]:
# ============================================================
# Weather aggregate helpers
# ============================================================

def norm_text(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).lower()
    s = s.replace("&", "and")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    # keep country/city/circuit core tokens, remove generic tokens
    s = re.sub(
        r"\b(grand|prix|gp|formula|f1|race|circuit|autodromo|autodrome|international|street|track)\b",
        " ",
        s,
    )
    s = re.sub(r"\s+", " ", s).strip()
    return s


def make_weather_aggregate(weather_raw: pd.DataFrame) -> pd.DataFrame:
    w = weather_raw.copy()

    required = [
        "Year", "Track", "AirTemp", "Humidity", "Pressure",
        "Rainfall", "TrackTemp", "WindSpeed",
    ]
    missing = [c for c in required if c not in w.columns]
    if missing:
        raise ValueError(f"weather missing columns: {missing}")

    w["Year"] = pd.to_numeric(w["Year"], errors="coerce").astype("Int64")
    w["Track_norm"] = w["Track"].map(norm_text)

    for c in ["AirTemp", "Humidity", "Pressure", "TrackTemp", "WindSpeed"]:
        w[c] = pd.to_numeric(w[c], errors="coerce")

    rainfall_num = pd.to_numeric(w["Rainfall"], errors="coerce")
    if rainfall_num.isna().all():
        rainfall_num = (
            w["Rainfall"].astype(str).str.lower()
            .isin(["true", "1", "yes", "y", "rain"])
            .astype(float)
        )
    w["Rainfall_num"] = rainfall_num.fillna(0.0)

    w["TrackTemp_minus_AirTemp"] = w["TrackTemp"] - w["AirTemp"]

    agg = (
        w.groupby(["Year", "Track_norm"], dropna=False)
        .agg(
            AirTemp_mean=("AirTemp", "mean"),
            AirTemp_min=("AirTemp", "min"),
            AirTemp_max=("AirTemp", "max"),
            AirTemp_std=("AirTemp", "std"),
            TrackTemp_mean=("TrackTemp", "mean"),
            TrackTemp_min=("TrackTemp", "min"),
            TrackTemp_max=("TrackTemp", "max"),
            TrackTemp_std=("TrackTemp", "std"),
            Humidity_mean=("Humidity", "mean"),
            Humidity_min=("Humidity", "min"),
            Humidity_max=("Humidity", "max"),
            Humidity_std=("Humidity", "std"),
            Pressure_mean=("Pressure", "mean"),
            Pressure_std=("Pressure", "std"),
            WindSpeed_mean=("WindSpeed", "mean"),
            WindSpeed_max=("WindSpeed", "max"),
            WindSpeed_std=("WindSpeed", "std"),
            Rainfall_mean=("Rainfall_num", "mean"),
            Rainfall_max=("Rainfall_num", "max"),
            Rainfall_sum=("Rainfall_num", "sum"),
            Rainfall_count=("Rainfall_num", "count"),
            TrackTemp_minus_AirTemp_mean=("TrackTemp_minus_AirTemp", "mean"),
            TrackTemp_minus_AirTemp_max=("TrackTemp_minus_AirTemp", "max"),
        )
        .reset_index()
    )

    agg["AirTemp_range"] = agg["AirTemp_max"] - agg["AirTemp_min"]
    agg["TrackTemp_range"] = agg["TrackTemp_max"] - agg["TrackTemp_min"]
    agg["Rainfall_any"] = (agg["Rainfall_max"] > 0).astype(np.int8)
    agg["Rainfall_rate"] = agg["Rainfall_sum"] / agg["Rainfall_count"].clip(lower=1)
    agg["WetRace"] = (agg["Rainfall_any"] > 0).astype(np.int8)

    key_cols = ["Year", "Track_norm"]
    feat_cols = [c for c in agg.columns if c not in key_cols]
    agg = agg[key_cols + feat_cols].rename(columns={c: f"W_{c}" for c in feat_cols})

    return agg


def build_race_to_track_map(race_values, weather_raw: pd.DataFrame) -> pd.DataFrame:
    track_values = sorted(weather_raw["Track"].dropna().astype(str).unique().tolist())
    track_norm_map = {norm_text(t): t for t in track_values}
    track_norms = sorted([k for k in track_norm_map.keys() if k])

    aliases = {
        "bahrain": ["bahrain", "sakhir"],
        "saudi arabian": ["jeddah", "saudi"],
        "australian": ["albert park", "melbourne", "australia"],
        "azerbaijan": ["baku", "azerbaijan"],
        "miami": ["miami"],
        "emilia romagna": ["imola", "emilia"],
        "monaco": ["monaco", "monte carlo"],
        "spanish": ["barcelona", "catalunya", "spain"],
        "canadian": ["gilles villeneuve", "montreal", "canada"],
        "austrian": ["red bull ring", "spielberg", "austria"],
        "british": ["silverstone", "great britain", "britain"],
        "hungarian": ["hungaroring", "hungary"],
        "belgian": ["spa", "francorchamps", "belgium"],
        "dutch": ["zandvoort", "netherlands"],
        "italian": ["monza", "italy"],
        "singapore": ["marina bay", "singapore"],
        "japanese": ["suzuka", "japan"],
        "qatar": ["lusail", "qatar"],
        "united states": ["americas", "austin", "cota", "united states"],
        "mexico": ["hermanos rodriguez", "mexico"],
        "mexico city": ["hermanos rodriguez", "mexico"],
        "sao paulo": ["interlagos", "sao paulo", "brazil"],
        "brazilian": ["interlagos", "sao paulo", "brazil"],
        "las vegas": ["las vegas", "vegas"],
        "abu dhabi": ["yas marina", "abu dhabi"],
        "chinese": ["shanghai", "china"],
        "french": ["paul ricard", "france"],
        "portuguese": ["portimao", "portugal"],
        "turkish": ["istanbul", "turkey"],
        "styrian": ["red bull ring", "spielberg", "austria"],
        "eifel": ["nurburgring", "eifel"],
        "tuscan": ["mugello", "mugello"],
    }

    def find_by_tokens(tokens):
        for token in tokens:
            tn = norm_text(token)
            for trn in track_norms:
                if tn and (tn in trn or trn in tn):
                    return trn
        return None

    rows = []
    for race in sorted(pd.Series(race_values).dropna().astype(str).unique().tolist()):
        rn = norm_text(race)
        best_track_norm = None
        method = "unmatched"

        # Direct substring.
        for trn in track_norms:
            if trn and (trn in rn or rn in trn):
                best_track_norm = trn
                method = "direct"
                break

        # Alias.
        if best_track_norm is None:
            for key, tokens in aliases.items():
                keyn = norm_text(key)
                if keyn and keyn in rn:
                    hit = find_by_tokens(tokens)
                    if hit is not None:
                        best_track_norm = hit
                        method = "alias"
                        break

        # Fuzzy fallback.
        if best_track_norm is None and track_norms:
            scores = [(SequenceMatcher(None, rn, trn).ratio(), trn) for trn in track_norms]
            score, trn = max(scores, key=lambda x: x[0])
            if score >= 0.42:
                best_track_norm = trn
                method = f"fuzzy_{score:.3f}"

        rows.append(
            {
                "Race": race,
                "Race_norm": rn,
                "Track_norm": best_track_norm,
                "Track_weather_name": track_norm_map.get(best_track_norm, None),
                "match_method": method,
            }
        )

    return pd.DataFrame(rows)


def add_weather_features(df: pd.DataFrame, race_map: pd.DataFrame, weather_agg: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.merge(race_map[["Race", "Track_norm"]], on="Race", how="left")

    # Year may be downcasted later; merge before reduce_mem_usage.
    out["Year"] = pd.to_numeric(out["Year"], errors="coerce").astype("Int64")
    out = out.merge(weather_agg, on=["Year", "Track_norm"], how="left")

    weather_cols = [c for c in out.columns if c.startswith("W_")]
    out["WeatherMissing"] = out[weather_cols].isna().all(axis=1).astype(np.int8)

    # Fill by Year mean first, then global mean.
    for c in weather_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
        out[c] = out.groupby("Year")[c].transform(lambda s: s.fillna(s.mean()))
        out[c] = out[c].fillna(out[c].mean())
        out[c] = out[c].fillna(0.0)

    # Helper key is not a model feature.
    out = out.drop(columns=["Track_norm"], errors="ignore")
    out["Year"] = out["Year"].astype(int)

    return out




In [39]:
# ============================================================
# Load data
# ============================================================

print_section("Load Data")

train_raw = pd.read_csv(CFG.TRAIN_PATH)
test_raw = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)
orig_raw = pd.read_csv(CFG.ORIG_PATH)

weather_path = find_weather_path()
weather_raw = pd.read_csv(weather_path)

print("train_raw:", train_raw.shape)
print("test_raw :", test_raw.shape)
print("orig_raw :", orig_raw.shape)
print("weather  :", weather_raw.shape, weather_path)
print("target mean competition:", train_raw[CFG.TARGET].mean())
print("target mean original   :", orig_raw[CFG.TARGET].mean())

assert CFG.ID_COL in train_raw.columns
assert CFG.ID_COL in test_raw.columns
assert CFG.TARGET in train_raw.columns
assert CFG.TARGET in orig_raw.columns
assert CFG.DANGER_COL in orig_raw.columns

if CFG.DANGER_COL in orig_raw.columns:
    orig_raw = orig_raw.drop(columns=[CFG.DANGER_COL])

# Weather is joined before normal FE. No target is used.
weather_agg = make_weather_aggregate(weather_raw)

race_values = pd.concat(
    [
        train_raw["Race"],
        test_raw["Race"],
        orig_raw["Race"] if "Race" in orig_raw.columns else pd.Series([], dtype=str),
    ],
    axis=0,
)
race_to_track = build_race_to_track_map(race_values, weather_raw)
race_to_track.to_csv(CFG.OUTDIR / "race_to_track_mapping.csv", index=False)
weather_agg.to_csv(CFG.OUTDIR / "weather_aggregate_year_track.csv", index=False)

print("weather_agg:", weather_agg.shape)
print("Race -> Track mapping:")
display(race_to_track)

unmatched = race_to_track[race_to_track["Track_norm"].isna()]
if len(unmatched) > 0:
    print("WARNING: unmatched Race values")
    display(unmatched)

train_raw = add_weather_features(train_raw, race_to_track, weather_agg)
test_raw = add_weather_features(test_raw, race_to_track, weather_agg)
orig_raw = add_weather_features(orig_raw, race_to_track, weather_agg)

weather_feature_cols = [c for c in train_raw.columns if c.startswith("W_")]
print("weather feature count:", len(weather_feature_cols))
print("WeatherMissing train/test/orig:",
      train_raw["WeatherMissing"].mean(),
      test_raw["WeatherMissing"].mean(),
      orig_raw["WeatherMissing"].mean())

train_raw = reduce_mem_usage(train_raw)
test_raw = reduce_mem_usage(test_raw)
orig_raw = reduce_mem_usage(orig_raw)

test_ids = test_raw[CFG.ID_COL].copy()
train_ids = train_raw[CFG.ID_COL].copy()





Load Data
train_raw: (439140, 16)
test_raw : (188165, 15)
orig_raw : (101371, 16)
weather  : (14896, 11) /kaggle/input/playground-series-s6-e5-weather-conditions/F1_Weather_2022_2025.csv
target mean competition: 0.19898210137996994
target mean original   : 0.25479673673930414
weather_agg: (92, 30)
Race -> Track mapping:


,Race,Race_norm,Track_norm,Track_weather_name,match_method
0,Abu Dhabi Grand Prix,abu dhabi,budapest,Budapest,fuzzy_0.471
1,Australian Grand Prix,australian,melbourne,Melbourne,alias
2,Austrian Grand Prix,austrian,spielberg,Spielberg,alias
3,Azerbaijan Grand Prix,azerbaijan,baku,Baku,alias
4,Bahrain Grand Prix,bahrain,sakhir,Sakhir,alias
5,Belgian Grand Prix,belgian,spa francorchamps,Spa-Francorchamps,alias
6,British Grand Prix,british,silverstone,Silverstone,alias
7,Canadian Grand Prix,canadian,shanghai,Shanghai,fuzzy_0.500
8,Chinese Grand Prix,chinese,shanghai,Shanghai,alias
9,Dutch Grand Prix,dutch,zandvoort,Zandvoort,alias


,Race,Race_norm,Track_norm,Track_weather_name,match_method
21,Pre-Season Track Session,pre season session,None,None,unmatched


weather feature count: 28
WeatherMissing train/test/orig: 0.09400191282962153 0.09381659713549279 0.0974144479190301


In [40]:
# ============================================================
# 020: Progress-window features
# ============================================================

def add_progress_window_base_features(df: pd.DataFrame, bins: int = 20) -> pd.DataFrame:
    """
    Add RaceProgress window features.

    This function does not use target.
    It is safe to apply to train/test/original before CV.

    Important:
      - Do not add Year here.
      - Do not add Race_Year_RaceProgressBin.
      - Keep bins coarse enough to avoid overfitting.
    """
    out = df.copy()

    rp = pd.to_numeric(out["RaceProgress"], errors="coerce").clip(0, 1)

    # 0..bins-1
    bin_id = np.floor(rp * bins).astype("float")
    bin_id = bin_id.clip(0, bins - 1).fillna(-1).astype(int)

    out[f"RaceProgressBin_{bins}"] = bin_id.astype(str)

    out[f"Race_RaceProgressBin_{bins}"] = (
        out["Race"].astype(str) + "_" + out[f"RaceProgressBin_{bins}"].astype(str)
    )

    out[f"Race_Compound_RaceProgressBin_{bins}"] = (
        out["Race"].astype(str)
        + "_"
        + out["Compound"].astype(str)
        + "_"
        + out[f"RaceProgressBin_{bins}"].astype(str)
    )

    return out





In [41]:
# ============================================================
# 025 component: Year/Stint flags
# ============================================================

def add_year_stint_flags_025(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add explicit Year/Stint/Tyre-wear features.

    This function does not use target.
    It intentionally does NOT reconstruct Normalized_TyreLife or any endpoint proxy.
    """
    out = df.copy()

    if "Year" in out.columns:
        out["Is_2022_025"] = (out["Year"] == 2022).astype(np.int8)
        out["Is_2023_025"] = (out["Year"] == 2023).astype(np.int8)
        out["Is_2024_025"] = (out["Year"] == 2024).astype(np.int8)
        out["Is_2025_025"] = (out["Year"] == 2025).astype(np.int8)

    if "Stint" in out.columns:
        out["Is_Stint1_025"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Stint2_025"] = (out["Stint"] == 2).astype(np.int8)
        out["Is_Stint3_025"] = (out["Stint"] == 3).astype(np.int8)
        out["Is_Stint4Plus_025"] = (out["Stint"] >= 4).astype(np.int8)
        out["Is_Second_Stint_025"] = (out["Stint"] == 2).astype(np.int8)
        out["Is_Third_StintPlus_025"] = (out["Stint"] >= 3).astype(np.int8)

    if "TyreLife" in out.columns:
        tyre = pd.to_numeric(out["TyreLife"], errors="coerce")
        tyre_clip = tyre.clip(lower=0)
        out["TyreLife_sq_025"] = (tyre ** 2).astype(np.float32)
        out["TyreLife_sqrt_025"] = np.sqrt(tyre_clip).astype(np.float32)
        out["TyreLife_log1p_025"] = np.log1p(tyre_clip).astype(np.float32)

    if "LapTime_Delta" in out.columns:
        out["LapTime_Delta_abs_025"] = pd.to_numeric(out["LapTime_Delta"], errors="coerce").abs().astype(np.float32)

    if "Cumulative_Degradation" in out.columns:
        out["Cumulative_Degradation_abs_025"] = pd.to_numeric(out["Cumulative_Degradation"], errors="coerce").abs().astype(np.float32)

    if {"Stint", "RaceProgress"}.issubset(out.columns):
        out["Stint_x_RaceProgress_025"] = (
            pd.to_numeric(out["Stint"], errors="coerce") *
            pd.to_numeric(out["RaceProgress"], errors="coerce")
        ).astype(np.float32)

    if {"Stint", "TyreLife"}.issubset(out.columns):
        out["Stint_x_TyreLife_sq_025"] = (
            pd.to_numeric(out["Stint"], errors="coerce") *
            (pd.to_numeric(out["TyreLife"], errors="coerce") ** 2)
        ).astype(np.float32)

    if {"TyreLife", "RaceProgress"}.issubset(out.columns):
        out["TyreLife_sq_x_RaceProgress_025"] = (
            (pd.to_numeric(out["TyreLife"], errors="coerce") ** 2) *
            pd.to_numeric(out["RaceProgress"], errors="coerce")
        ).astype(np.float32)

    return out


YEAR_STINT_FLAG_FEATURES_025 = [
    "Is_2022_025", "Is_2023_025", "Is_2024_025", "Is_2025_025",
    "Is_Stint1_025", "Is_Stint2_025", "Is_Stint3_025", "Is_Stint4Plus_025",
    "Is_Second_Stint_025", "Is_Third_StintPlus_025",
    "TyreLife_sq_025", "TyreLife_sqrt_025", "TyreLife_log1p_025",
    "LapTime_Delta_abs_025", "Cumulative_Degradation_abs_025",
    "Stint_x_RaceProgress_025", "Stint_x_TyreLife_sq_025", "TyreLife_sq_x_RaceProgress_025",
]

YEAR_STINT_CROSS_FEATURES_025 = [
    "Year_Stint_025",
    "Year_Compound_025",
]



In [42]:
# ============================================================
# shared_003-style FE
# ============================================================

BASE_CAT_COLS = ["Driver", "Compound", "Race"]

NUMERIC_BASE_COLS = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]





In [43]:
# ============================================================
# 040: RealMLP shared001v2-style feature representation
# ============================================================

SHARED001V2_FLOOR_CAT_COLS_040 = [
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]

SHARED001V2_BIN_COLS_040 = [
    "RaceProgress_200_quantile_bin_",
    "LapTime (s)_7_quantile_bin_",
]

SHARED001V2_COUNT_SOURCE_COLS_040 = [
    "PitStop_cat_",
    "RaceProgress_200_quantile_bin_",
    "LapTime (s)_7_quantile_bin_",
    "Race_Compound",
    "Race_Year",
    "Compound",
    "Race",
    "Driver",
]


def _safe_qbin_040(s: pd.Series, q: int, name: str) -> pd.Series:
    """Robust quantile binning used only as an unsupervised representation."""
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return pd.Series([f"{name}_missing"] * len(s), index=s.index, dtype="object")

    # qcut can fail with many duplicate values. Ranking first preserves ordering
    # while keeping the operation target-free.
    r = x.rank(method="first")
    try:
        b = pd.qcut(r, q=q, labels=False, duplicates="drop")
    except Exception:
        b = pd.cut(r, bins=min(q, max(int(r.nunique()), 1)), labels=False)

    return (
        pd.Series(b, index=s.index)
        .astype("Int64")
        .astype(str)
        .replace("<NA>", f"{name}_missing")
        .map(lambda z: f"{name}_{z}")
        .astype(str)
    )


def add_shared001v2_features_040(all_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add 029/038-style representation features to the LGB branch.

    This block is target-free:
    - floor/category conversion is row-wise
    - quantile bins are unsupervised
    - counts are fitted on train+orig+test combined, as count/frequency encodings
      in this notebook already use combined availability.
    """
    out = all_df.copy()

    if not CFG.USE_SHARED001V2_FEATURES_040:
        return out

    # 029-family keeps raw PitStop and also uses a categorical PitStop view.
    if "PitStop" in out.columns:
        pit = pd.to_numeric(out["PitStop"], errors="coerce").fillna(-1)
        out["PitStop_cat_"] = np.floor(pit).astype(np.int16).astype(str)

    # Floored numerical categories. These are intended for tree splits and count encodings.
    for col in SHARED001V2_FLOOR_CAT_COLS_040:
        if col not in out.columns:
            continue
        x = pd.to_numeric(out[col], errors="coerce")
        cat_col = f"{col}_cat_"
        out[cat_col] = (
            np.floor(x.fillna(-9999))
            .astype(np.int32)
            .astype(str)
        )
        if cat_col not in SHARED001V2_COUNT_SOURCE_COLS_040:
            SHARED001V2_COUNT_SOURCE_COLS_040.append(cat_col)

    if "RaceProgress" in out.columns:
        out["RaceProgress_200_quantile_bin_"] = _safe_qbin_040(
            out["RaceProgress"], q=200, name="RP200"
        )

    if "LapTime (s)" in out.columns:
        out["LapTime (s)_7_quantile_bin_"] = _safe_qbin_040(
            out["LapTime (s)"], q=7, name="LT7"
        )

    # Count encoding for 029-family categorical/bin views.
    count_cols = [
        c for c in SHARED001V2_COUNT_SOURCE_COLS_040
        if c in out.columns
    ]

    for col in count_cols:
        vc = out[col].astype(str).value_counts(dropna=False)
        if col == "PitStop_cat_":
            count_name = "_PitStop_cat_count"
        else:
            count_name = f"{col}_count_040"

        out[count_name] = (
            out[col].astype(str).map(vc).fillna(0).astype(np.float32)
        )

    return out


def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    for col in BASE_CAT_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

    # shared_003 baseline FE style
    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedTotalLaps"] = safe_divide(
            out["LapNumber"],
            out["RaceProgress"].clip(lower=eps),
            eps,
        ).replace([np.inf, -np.inf], np.nan).clip(1, 120)

        out["LapsRemaining"] = (
            out["EstimatedTotalLaps"] - out["LapNumber"]
        ).clip(lower=0)

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreAgeRatio"] = safe_divide(
            out["TyreLife"],
            out["LapNumber"].clip(lower=1),
            eps,
        )

    if {"Cumulative_Degradation", "TyreLife"}.issubset(out.columns):
        out["DegPerTyreLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if {"Position", "RaceProgress"}.issubset(out.columns):
        out["PositionPressure"] = out["Position"] * out["RaceProgress"]

    # Additional compact full-FE features from shared_003 description
    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["LapProgress_x_LapNumber"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["LapPerTyreLife"] = safe_divide(out["LapNumber"], out["TyreLife"] + 1, eps)
        out["LapMinusTyreLife"] = out["LapNumber"] - out["TyreLife"]
        out["TyreLifeMinusLap"] = out["TyreLife"] - out["LapNumber"]

    if {"TyreLife", "EstimatedTotalLaps"}.issubset(out.columns):
        out["TyreAgeVsRace"] = safe_divide(
            out["TyreLife"],
            out["EstimatedTotalLaps"].clip(lower=1),
            eps,
        )

    if {"TyreLife", "RaceProgress"}.issubset(out.columns):
        out["PitWindowPressure"] = out["TyreLife"] * out["RaceProgress"]
        out["TyreLife_x_RaceProgress"] = out["TyreLife"] * out["RaceProgress"]

    if {"Cumulative_Degradation", "LapNumber"}.issubset(out.columns):
        out["DegPerRaceLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["LapNumber"].clip(lower=1),
            eps,
        )

    if {"LapTime_Delta", "TyreLife"}.issubset(out.columns):
        out["DeltaPerTyreLap"] = safe_divide(
            out["LapTime_Delta"],
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if "LapTime_Delta" in out.columns:
        out["DeltaAbs"] = out["LapTime_Delta"].abs()
        out["LapTimeDeltaPositive"] = (out["LapTime_Delta"] > 0).astype(np.int8)
        out["LapTimeDeltaNegative"] = (out["LapTime_Delta"] < 0).astype(np.int8)

    if "Cumulative_Degradation" in out.columns:
        out["Abs_Cumulative_Degradation"] = out["Cumulative_Degradation"].abs()
        out["Positive_Degradation"] = (out["Cumulative_Degradation"] > 0).astype(np.int8)

    if "Position_Change" in out.columns:
        out["Abs_Position_Change"] = out["Position_Change"].abs()
        out["Gained_Position"] = (out["Position_Change"] > 0).astype(np.int8)
        out["Lost_Position"] = (out["Position_Change"] < 0).astype(np.int8)

    if {"Stint", "TyreLife"}.issubset(out.columns):
        out["StintPressure"] = out["Stint"] * out["TyreLife"]
        out["TyreLife_x_Stint"] = out["TyreLife"] * out["Stint"]

    if {"Stint", "LapNumber"}.issubset(out.columns):
        out["Stint_x_LapNumber"] = out["Stint"] * out["LapNumber"]
        out["Is_First_Stint"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Late_Stint"] = (out["Stint"] >= 3).astype(np.int8)

    if "RaceProgress" in out.columns:
        out["Early_Race"] = (out["RaceProgress"] <= 0.25).astype(np.int8)
        out["Mid_Race"] = (
            (out["RaceProgress"] > 0.25) &
            (out["RaceProgress"] <= 0.65)
        ).astype(np.int8)
        out["Late_Race"] = (out["RaceProgress"] > 0.65).astype(np.int8)

        out["RaceProgressBin"] = pd.cut(
            out["RaceProgress"],
            bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
            labels=["P1", "P2", "P3", "P4", "P5"],
        ).astype(str)

    if "TyreLife" in out.columns:
        out["TyreLifeBin"] = pd.cut(
            out["TyreLife"],
            bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
            labels=["T_000_003", "T_004_007", "T_008_012", "T_013_020", "T_021_030", "T_031_plus"],
        ).astype(str)

    if "Year" in out.columns:
        out["Year_str"] = out["Year"].astype(str)
        out["is_2023"] = (out["Year"] == 2023).astype(np.int8)
        out["Year_minus_2022"] = (out["Year"] - 2022).astype(np.int8)

    # 026 includes the 025 explicit Year/Stint/Tyre-wear feature block.
    if CFG.USE_YEAR_STINT_FLAGS_025:
        out = add_year_stint_flags_025(out)

    return out


def add_interaction_categories(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def make_cross(name, cols):
        if set(cols).issubset(out.columns):
            s = out[cols[0]].astype(str)
            for c in cols[1:]:
                s = s + "_" + out[c].astype(str)
            out[name] = s

    make_cross("Driver_Compound", ["Driver", "Compound"])
    make_cross("Race_Compound", ["Race", "Compound"])
    make_cross("Race_Year", ["Race", "Year"])
    make_cross("Driver_Race", ["Driver", "Race"])
    make_cross("Driver_Year", ["Driver", "Year"])
    make_cross("Stint_Compound", ["Stint", "Compound"])
    make_cross("Compound_TyreLifeBin", ["Compound", "TyreLifeBin"])
    make_cross("Compound_RaceProgressBin", ["Compound", "RaceProgressBin"])

    # 026 includes the 025 minimal Year/Stint categorical crosses.
    # Do NOT add Race_Year_Stint or Driver_Year_Stint in this merged pass.
    if CFG.ADD_YEAR_STINT_CROSSES_025:
        make_cross("Year_Stint_025", ["Year", "Stint"])
        make_cross("Year_Compound_025", ["Year", "Compound"])

    return out


def add_frequency_features(all_df: pd.DataFrame) -> pd.DataFrame:
    out = all_df.copy()

    freq_cols = [
        "Driver",
        "Race",
        "Compound",
        "Driver_Race",
        "Race_Year",
        "Year_str",
    ]

    # 026 includes frequency only for the 025 Year/Stint crosses.
    # No target encoding for these added crosses in this first merged pass.
    if CFG.ADD_YEAR_STINT_CROSSES_025:
        freq_cols += [
            "Year_Stint_025",
            "Year_Compound_025",
        ]

    freq_cols = [c for c in freq_cols if c in out.columns]

    total = len(out)

    for col in freq_cols:
        counts = out[col].astype(str).value_counts(dropna=False)
        out[f"{col}_freq"] = (
            out[col].astype(str).map(counts).fillna(0) / total
        ).astype(np.float32)

    return out


def add_lag_features(all_df: pd.DataFrame) -> pd.DataFrame:
    out = all_df.copy()

    required = {"Driver", "Race", "Stint", "LapNumber"}
    if not required.issubset(out.columns):
        return out

    out["_orig_order_for_lag"] = np.arange(len(out))
    out = out.sort_values(["Driver", "Race", "Stint", "LapNumber", "_orig_order_for_lag"])

    group_cols = ["Driver", "Race", "Stint"]

    if "LapTime_Delta" in out.columns:
        out["Delta_lag1"] = (
            out.groupby(group_cols, sort=False)["LapTime_Delta"]
            .shift(1)
        )

        out["Delta_roll3_mean"] = (
            out.groupby(group_cols, sort=False)["LapTime_Delta"]
            .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
        )

    if "Cumulative_Degradation" in out.columns:
        out["Deg_diff"] = (
            out.groupby(group_cols, sort=False)["Cumulative_Degradation"]
            .diff()
        )

    if "TyreLife" in out.columns:
        out["TyreLife_growth"] = (
            out.groupby(group_cols, sort=False)["TyreLife"]
            .diff()
        )

    if "LapTime (s)" in out.columns:
        out["LapTime_lag1"] = (
            out.groupby(group_cols, sort=False)["LapTime (s)"]
            .shift(1)
        )

        out["LapTime_diff"] = (
            out.groupby(group_cols, sort=False)["LapTime (s)"]
            .diff()
        )

    out = out.sort_values("_orig_order_for_lag").drop(columns=["_orig_order_for_lag"])

    return out


def fill_missing_and_types(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan)

    for col in out.columns:
        if col in [CFG.ID_COL, CFG.TARGET, "_dataset"]:
            continue

        if out[col].dtype == "object" or str(out[col].dtype).startswith("category") or str(out[col].dtype).startswith("string"):
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)
        else:
            med = out[col].median()
            out[col] = out[col].fillna(med)
            if out[col].dtype == "float64":
                out[col] = out[col].astype(np.float32)

    return reduce_mem_usage(out)


print_section("Feature Engineering")

train_comp = train_raw.copy()
test_comp = test_raw.copy()
orig_comp = orig_raw.copy()

train_comp["_dataset"] = "train"
test_comp["_dataset"] = "test"
orig_comp["_dataset"] = "orig"

# original has no id in the reference dataset in many versions.
# Add synthetic negative ids only for alignment.
if CFG.ID_COL not in orig_comp.columns:
    orig_comp[CFG.ID_COL] = -np.arange(1, len(orig_comp) + 1)

# Add missing target to test for concat.
test_comp[CFG.TARGET] = np.nan

all_df = pd.concat(
    [train_comp, orig_comp, test_comp],
    axis=0,
    ignore_index=True,
    sort=False,
)

print("all_df initial:", all_df.shape)

all_df = add_base_features(all_df)
all_df = add_interaction_categories(all_df)
all_df = add_shared001v2_features_040(all_df)
all_df = add_frequency_features(all_df)
all_df = add_lag_features(all_df)
all_df = fill_missing_and_types(all_df)

print("all_df fe:", all_df.shape)

train_fe = all_df[all_df["_dataset"] == "train"].reset_index(drop=True).copy()
orig_fe = all_df[all_df["_dataset"] == "orig"].reset_index(drop=True).copy()
test_fe = all_df[all_df["_dataset"] == "test"].reset_index(drop=True).copy()




Feature Engineering
all_df initial: (728676, 46)
all_df fe: (728676, 152)


In [44]:
# ============================================================
# 020差分: RaceProgress window base features
# ============================================================

train_fe = add_progress_window_base_features(train_fe, bins=CFG.PROGRESS_BINS)
orig_fe = add_progress_window_base_features(orig_fe, bins=CFG.PROGRESS_BINS)
test_fe = add_progress_window_base_features(test_fe, bins=CFG.PROGRESS_BINS)

print("020 progress window base columns:")
for c in CFG.PROGRESS_WINDOW_CAT_COLS:
    print(
        c,
        "train:", c in train_fe.columns,
        "orig:", c in orig_fe.columns,
        "test:", c in test_fe.columns,
    )

# Guardrails: do not add Year interaction for this experiment
assert "Race_Year_RaceProgressBin_20" not in train_fe.columns
assert "TE_Race_Year_RaceProgressBin_20" not in train_fe.columns

# Restore target dtypes
train_fe[CFG.TARGET] = train_raw[CFG.TARGET].astype(int).values
orig_fe[CFG.TARGET] = orig_raw[CFG.TARGET].astype(int).values

print("train_fe:", train_fe.shape)
print("orig_fe :", orig_fe.shape)
print("test_fe :", test_fe.shape)




020 progress window base columns:
RaceProgressBin_20 train: True orig: True test: True
Race_RaceProgressBin_20 train: True orig: True test: True
Race_Compound_RaceProgressBin_20 train: True orig: True test: True
train_fe: (439140, 155)
orig_fe : (101371, 155)
test_fe : (188165, 155)


In [45]:
# ============================================================
# Feature list and encoding
# ============================================================

CAT_COLS_FINAL = [
    "Driver",
    "Compound",
    "Race",
    "Year_str",
    "Driver_Compound",
    "Race_Compound",
    "Race_Year",
    "Driver_Race",
    "Driver_Year",
    "Compound_TyreLifeBin",
    "Compound_RaceProgressBin",
    "Stint_Compound",
]
CAT_COLS_FINAL = [c for c in CAT_COLS_FINAL if c in train_fe.columns and c in test_fe.columns]

# 020差分: add progress-window categorical columns
for c in CFG.PROGRESS_WINDOW_CAT_COLS:
    if c in train_fe.columns and c in test_fe.columns and c in orig_fe.columns:
        if c not in CAT_COLS_FINAL:
            CAT_COLS_FINAL.append(c)

# 026 includes 025 minimal Year/Stint categorical crosses as ordinal categories, not TE.
for c in YEAR_STINT_CROSS_FEATURES_025:
    if c in train_fe.columns and c in test_fe.columns and c in orig_fe.columns:
        if c not in CAT_COLS_FINAL:
            CAT_COLS_FINAL.append(c)

# 040: add 029/038-style categorical/bin features as ordinal categories.
SHARED001V2_CAT_FEATURES_040 = (
    ["PitStop_cat_"]
    + [f"{c}_cat_" for c in SHARED001V2_FLOOR_CAT_COLS_040]
    + SHARED001V2_BIN_COLS_040
)
for c in SHARED001V2_CAT_FEATURES_040:
    if c in train_fe.columns and c in test_fe.columns and c in orig_fe.columns:
        if c not in CAT_COLS_FINAL:
            CAT_COLS_FINAL.append(c)

print("020 CAT_COLS_FINAL added:", [c for c in CFG.PROGRESS_WINDOW_CAT_COLS if c in CAT_COLS_FINAL])
print("025 CAT_COLS_FINAL added:", [c for c in YEAR_STINT_CROSS_FEATURES_025 if c in CAT_COLS_FINAL])
print("040 CAT_COLS_FINAL added:", [c for c in SHARED001V2_CAT_FEATURES_040 if c in CAT_COLS_FINAL])

DROP_FROM_X = [
    CFG.ID_COL,
    CFG.TARGET,
    "_dataset",
]
# 040 intentionally keeps raw PitStop to mirror the strong 029/038 RealMLP feature set.
if not CFG.KEEP_RAW_PITSTOP_040:
    DROP_FROM_X.append("PitStop")

FEATURE_COLS_BASE = [
    c for c in train_fe.columns
    if c not in DROP_FROM_X and c in test_fe.columns
]

# Keep columns that also exist in original.
FEATURE_COLS_BASE = [
    c for c in FEATURE_COLS_BASE
    if c in orig_fe.columns
]

print("base feature count before TE:", len(FEATURE_COLS_BASE))
print("CAT_COLS_FINAL:", len(CAT_COLS_FINAL), CAT_COLS_FINAL)

# Ordinal encode selected categorical features.
oe = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=np.int32,
)

encode_base = pd.concat(
    [
        train_fe[CAT_COLS_FINAL],
        orig_fe[CAT_COLS_FINAL],
        test_fe[CAT_COLS_FINAL],
    ],
    axis=0,
    ignore_index=True,
).astype(str)

oe.fit(encode_base)

for df in [train_fe, orig_fe, test_fe]:
    df[CAT_COLS_FINAL] = oe.transform(df[CAT_COLS_FINAL].astype(str))

# Ensure all model features are numeric.
for df in [train_fe, orig_fe, test_fe]:
    for col in FEATURE_COLS_BASE:
        if df[col].dtype == "object" or str(df[col].dtype).startswith("category") or str(df[col].dtype).startswith("string"):
            codes, _ = pd.factorize(df[col].astype(str), sort=False)
            df[col] = codes.astype(np.int32)
        elif df[col].dtype == "float64":
            df[col] = df[col].astype(np.float32)




020 CAT_COLS_FINAL added: ['RaceProgressBin_20', 'Race_RaceProgressBin_20', 'Race_Compound_RaceProgressBin_20']
025 CAT_COLS_FINAL added: ['Year_Stint_025', 'Year_Compound_025']
040 CAT_COLS_FINAL added: ['PitStop_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_']
base feature count before TE: 152
CAT_COLS_FINAL: 29 ['Driver', 'Compound', 'Race', 'Year_str', 'Driver_Compound', 'Race_Compound', 'Race_Year', 'Driver_Race', 'Driver_Year', 'Compound_TyreLifeBin', 'Compound_RaceProgressBin', 'Stint_Compound', 'RaceProgressBin_20', 'Race_RaceProgressBin_20', 'Race_Compound_RaceProgressBin_20', 'Year_Stint_025', 'Year_Compound_025', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradati

In [46]:
# ============================================================
# Fold-safe Target Encoding
# ============================================================

TE_COLUMNS = [
    "Driver",
    "Race_Year",
    "Race_Compound",
    "Driver_Race",
    "Driver_Year",
    "Race",
    "Stint_Compound",
]
TE_COLUMNS = [c for c in TE_COLUMNS if c in train_fe.columns and c in test_fe.columns and c in orig_fe.columns]

# 026: do NOT add Year_Stint_025 / Year_Compound_025 to TE_COLUMNS at first pass.
if not CFG.ADD_YEAR_STINT_TE_025:
    TE_COLUMNS = [c for c in TE_COLUMNS if c not in YEAR_STINT_CROSS_FEATURES_025]

TE_NAMES = [f"TE_{c}" for c in TE_COLUMNS]

print("TE_COLUMNS:", TE_COLUMNS)


def add_fold_target_encoding(X_tr, y_tr, X_va, X_te, X_orig=None):
    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_te.copy()

    if X_orig is not None:
        X_orig = X_orig.copy()

    if not CFG.USE_TARGET_ENCODING or len(TE_COLUMNS) == 0:
        return X_tr, X_va, X_te, X_orig

    te = TargetEncoder(
        cv=5,
        smooth="auto",
        shuffle=True,
        random_state=CFG.SEED,
    )

    tr_encoded = te.fit_transform(X_tr[TE_COLUMNS], y_tr)
    va_encoded = te.transform(X_va[TE_COLUMNS])
    te_encoded = te.transform(X_te[TE_COLUMNS])

    X_tr[TE_NAMES] = tr_encoded
    X_va[TE_NAMES] = va_encoded
    X_te[TE_NAMES] = te_encoded

    if X_orig is not None:
        # Original rows are already included in X_tr in this implementation.
        # This branch is kept only for interface symmetry.
        pass

    return X_tr, X_va, X_te, X_orig




TE_COLUMNS: ['Driver', 'Race_Year', 'Race_Compound', 'Driver_Race', 'Driver_Year', 'Race', 'Stint_Compound']


In [47]:
# ============================================================
# 020: Fold-safe progress-window count/frequency + TE
# ============================================================

def add_fold_count_frequency_features(
    X_tr: pd.DataFrame,
    X_va: pd.DataFrame,
    X_te: pd.DataFrame,
    cols: list,
):
    """
    Add fold-safe count/frequency features.
    Count maps are fitted on X_tr only.
    """
    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_te.copy()

    added = []
    denom = max(len(X_tr), 1)

    for col in cols:
        if col not in X_tr.columns:
            print(f"[020 count/freq skip] {col} not in X_tr")
            continue

        key_tr = X_tr[col].astype(str)
        key_va = X_va[col].astype(str)
        key_te = X_te[col].astype(str)

        vc = key_tr.value_counts(dropna=False)

        cnt_col = f"{col}_fold_count"
        frq_col = f"{col}_fold_freq"

        X_tr[cnt_col] = key_tr.map(vc).fillna(0).astype(np.float32)
        X_va[cnt_col] = key_va.map(vc).fillna(0).astype(np.float32)
        X_te[cnt_col] = key_te.map(vc).fillna(0).astype(np.float32)

        X_tr[frq_col] = (X_tr[cnt_col] / denom).astype(np.float32)
        X_va[frq_col] = (X_va[cnt_col] / denom).astype(np.float32)
        X_te[frq_col] = (X_te[cnt_col] / denom).astype(np.float32)

        added.extend([cnt_col, frq_col])

    return X_tr, X_va, X_te, added


def add_fold_progress_window_target_encoding(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    X_va: pd.DataFrame,
    X_te: pd.DataFrame,
    cols: list,
    cv: int = 5,
    smooth="auto",
    seed: int = 3407,
):
    """
    Add fold-safe TargetEncoder features for progress-window categorical columns.
    Encoder is fit only on outer-fold X_tr/y_tr.
    """
    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_te.copy()

    use_cols = [c for c in cols if c in X_tr.columns]

    if len(use_cols) == 0:
        print("[020 TE skip] no progress-window TE columns found")
        return X_tr, X_va, X_te, []

    te = TargetEncoder(
        cv=cv,
        smooth=smooth,
        shuffle=True,
        random_state=seed,
    )

    out_names = [f"TE_{c}" for c in use_cols]

    X_tr[out_names] = te.fit_transform(X_tr[use_cols].astype(str), y_tr)
    X_va[out_names] = te.transform(X_va[use_cols].astype(str))
    X_te[out_names] = te.transform(X_te[use_cols].astype(str))

    for c in out_names:
        X_tr[c] = X_tr[c].astype(np.float32)
        X_va[c] = X_va[c].astype(np.float32)
        X_te[c] = X_te[c].astype(np.float32)

    return X_tr, X_va, X_te, out_names




In [48]:
# ============================================================
# CV setup
# ============================================================

print_section("CV Setup")

y_comp = train_fe[CFG.TARGET].astype(int).values
y_orig = orig_fe[CFG.TARGET].astype(int).values

strat_key = train_fe[CFG.TARGET].astype(str) + "__" + train_fe["Year"].astype(str)

folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(train_fe, strat_key)
)

for i, (_, va_idx) in enumerate(folds, 1):
    print(
        f"fold {i}: valid rows={len(va_idx)}, valid rate={train_fe.iloc[va_idx][CFG.TARGET].mean():.6f}"
    )


CV Setup
fold 1: valid rows=87828, valid rate=0.198991
fold 2: valid rows=87828, valid rate=0.198991
fold 3: valid rows=87828, valid rate=0.198968
fold 4: valid rows=87828, valid rate=0.198968
fold 5: valid rows=87828, valid rate=0.198991


In [ ]:
# ============================================================
# 047: LightGBM Optuna search
# ============================================================

def trial_lgb_params(trial: optuna.trial.Trial) -> dict:
    """Narrow-to-moderate search around the successful 040 LGB branch."""
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.010, 0.050, log=True),
        "num_leaves": trial.suggest_categorical("num_leaves", [15, 31, 63, 95, 127]),
        "min_child_samples": trial.suggest_int("min_child_samples", 40, 260),
        "subsample": trial.suggest_float("subsample", 0.70, 0.95),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.95),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 50.0, log=True),
        "max_bin": trial.suggest_categorical("max_bin", [127, 255]),
        "max_depth": trial.suggest_categorical("max_depth", [-1, 5, 7, 9]),
    }


def make_lgb_model_with_params(seed: int, device: str, params: dict):
    return lgb.LGBMClassifier(
        n_estimators=CFG.LGB_N_ESTIMATORS,
        learning_rate=params["learning_rate"],
        num_leaves=params["num_leaves"],
        min_child_samples=params["min_child_samples"],
        subsample=params["subsample"],
        subsample_freq=params.get("subsample_freq", 1),
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        max_bin=params["max_bin"],
        max_depth=params.get("max_depth", -1),
        class_weight="balanced",
        device=device,
        gpu_use_dp=False,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )


def fit_lgb_with_fallback_params(model, X_tr, y_tr, X_va, y_va, params: dict):
    try:
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[
                lgb.early_stopping(CFG.LGB_EARLY_STOPPING, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        return model, model.get_params().get("device", "unknown")
    except Exception as e:
        print("LightGBM GPU/device training failed. Retrying with CPU.")
        print("Error:", repr(e))
        cpu_model = make_lgb_model_with_params(
            seed=model.get_params().get("random_state", CFG.SEED),
            device="cpu",
            params=params,
        )
        cpu_model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[
                lgb.early_stopping(CFG.LGB_EARLY_STOPPING, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        return cpu_model, "cpu"


def make_fold_matrices(tr_idx, va_idx, fold: int):
    X_tr_comp = train_fe.iloc[tr_idx][FEATURE_COLS_BASE].copy()
    y_tr_comp = train_fe.iloc[tr_idx][CFG.TARGET].astype(int).reset_index(drop=True)

    X_va = train_fe.iloc[va_idx][FEATURE_COLS_BASE].copy()
    y_va = train_fe.iloc[va_idx][CFG.TARGET].astype(int).values

    if CFG.USE_ORIGINAL_ROWS:
        X_orig_fold = orig_fe[FEATURE_COLS_BASE].copy()
        y_orig_fold = orig_fe[CFG.TARGET].astype(int).reset_index(drop=True)
        X_tr = pd.concat(
            [X_tr_comp.reset_index(drop=True), X_orig_fold.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
        y_tr = pd.concat([y_tr_comp, y_orig_fold], axis=0, ignore_index=True)
    else:
        X_tr = X_tr_comp.reset_index(drop=True)
        y_tr = y_tr_comp

    X_te = test_fe[FEATURE_COLS_BASE].copy()

    X_tr, X_va, X_te, _ = add_fold_target_encoding(
        X_tr=X_tr,
        y_tr=y_tr,
        X_va=X_va,
        X_te=X_te,
        X_orig=None,
    )

    progress_features = []
    if CFG.USE_PROGRESS_WINDOW_TE:
        X_tr, X_va, X_te, progress_count_features = add_fold_count_frequency_features(
            X_tr=X_tr,
            X_va=X_va,
            X_te=X_te,
            cols=CFG.PROGRESS_WINDOW_CAT_COLS,
        )
        X_tr, X_va, X_te, progress_te_features = add_fold_progress_window_target_encoding(
            X_tr=X_tr,
            y_tr=y_tr,
            X_va=X_va,
            X_te=X_te,
            cols=CFG.PROGRESS_WINDOW_TE_COLS,
            cv=5,
            smooth="auto",
            seed=CFG.SEED,
        )
        progress_features = progress_count_features + progress_te_features

        assert "Race_Year_RaceProgressBin_20" not in X_tr.columns
        assert "TE_Race_Year_RaceProgressBin_20" not in X_tr.columns
        assert "Race_RaceProgressBin_20" in X_tr.columns
        assert "Race_Compound_RaceProgressBin_20" in X_tr.columns
        assert "TE_Race_RaceProgressBin_20" in X_tr.columns
        assert "TE_Race_Compound_RaceProgressBin_20" in X_tr.columns

    return X_tr, y_tr, X_va, y_va, progress_features


print_section("Optuna Search: LGB 040 shared001v2 features")
print("Baseline 040 CV:", CFG.BASELINE_040_CV)
print("Optuna seeds:", CFG.OPTUNA_SEEDS)
print("N_TRIALS:", CFG.OPTUNA_N_TRIALS)
print("TIMEOUT_SEC:", CFG.OPTUNA_TIMEOUT_SEC)

trial_records = []
first_final_features = None
first_progress_features = []

def objective(trial: optuna.trial.Trial) -> float:
    global first_final_features, first_progress_features

    params = trial_lgb_params(trial)
    oof_trial = np.zeros(len(train_fe), dtype=np.float32)
    fold_scores_trial = []
    fold_seed_scores_trial = []
    best_iterations_trial = []
    used_devices_trial = []

    for fold, (tr_idx, va_idx) in enumerate(folds, 1):
        X_tr, y_tr, X_va, y_va, progress_features = make_fold_matrices(tr_idx, va_idx, fold)

        if first_final_features is None:
            first_final_features = X_tr.columns.tolist()
            first_progress_features = progress_features
            print("final feature count:", len(first_final_features))
            print("final features:", first_final_features)

        fold_val = np.zeros(len(va_idx), dtype=np.float32)

        for seed in CFG.OPTUNA_SEEDS:
            model = make_lgb_model_with_params(seed=seed, device=CFG.LGB_DEVICE, params=params)
            model, used_device = fit_lgb_with_fallback_params(
                model=model,
                X_tr=X_tr,
                y_tr=y_tr,
                X_va=X_va,
                y_va=y_va,
                params=params,
            )
            p_va = model.predict_proba(X_va)[:, 1].astype(np.float32)
            fold_val += p_va / len(CFG.OPTUNA_SEEDS)

            seed_auc = roc_auc_score(y_va, p_va)
            fold_seed_scores_trial.append({
                "fold": fold,
                "seed": seed,
                "auc": float(seed_auc),
                "best_iteration": int(getattr(model, "best_iteration_", -1) or -1),
                "used_device": used_device,
            })
            best_iterations_trial.append(int(getattr(model, "best_iteration_", -1) or -1))
            used_devices_trial.append(used_device)

            del model
            gc.collect()

        oof_trial[va_idx] = fold_val
        fold_auc = roc_auc_score(y_va, fold_val)
        fold_scores_trial.append(float(fold_auc))

        del X_tr, X_va
        gc.collect()

    cv_auc_trial = roc_auc_score(y_comp, oof_trial)

    trial.set_user_attr("fold_scores", fold_scores_trial)
    trial.set_user_attr("fold_mean", float(np.mean(fold_scores_trial)))
    trial.set_user_attr("fold_std", float(np.std(fold_scores_trial)))
    trial.set_user_attr("fold_seed_scores", fold_seed_scores_trial)
    trial.set_user_attr("best_iterations", best_iterations_trial)
    trial.set_user_attr("used_devices", sorted(set(used_devices_trial)))

    print(f"trial {trial.number}: cv={cv_auc_trial:.10f}, params={params}")
    return float(cv_auc_trial)


sampler = optuna.samplers.TPESampler(seed=CFG.SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(
    objective,
    n_trials=CFG.OPTUNA_N_TRIALS,
    timeout=CFG.OPTUNA_TIMEOUT_SEC,
    gc_after_trial=True,
)

best_trial = study.best_trial
best_params = dict(best_trial.params)
best_value = float(best_trial.value)

print_section("Optuna Result")
print("best_value:", best_value)
print("best_trial:", best_trial.number)
print("best_params:", best_params)
print("delta_vs_040_cv:", best_value - CFG.BASELINE_040_CV)

# Save feature list observed during the objective.
if first_final_features is None:
    raise RuntimeError("No successful Optuna trial; no feature list captured.")

feature_df = pd.DataFrame({
    "feature": first_final_features,
    "is_cat_encoded": [c in CAT_COLS_FINAL for c in first_final_features],
    "is_te": [c in TE_NAMES for c in first_final_features],
    "is_weather": [c.startswith("W_") or c == "WeatherMissing" for c in first_final_features],
    "is_025_year_stint": [c.endswith("_025") for c in first_final_features],
    "is_040_shared001v2_cat": [c in SHARED001V2_CAT_FEATURES_040 for c in first_final_features],
    "is_040_shared001v2_count": [c == "_PitStop_cat_count" or c.endswith("_count_040") for c in first_final_features],
    "is_040_shared001v2_bin": [c in SHARED001V2_BIN_COLS_040 for c in first_final_features],
    "is_progress_window_any": [(c in CFG.PROGRESS_WINDOW_CAT_COLS or c in first_progress_features) for c in first_final_features],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

# Guardrails.
assert CFG.DANGER_COL not in first_final_features
assert "log1p_abs_LapTime_Delta" not in first_final_features
assert "signed_log1p_abs_LapTime_Delta" not in first_final_features
assert "Race_Year_RaceProgressBin_20" not in first_final_features
assert "TE_Race_Year_RaceProgressBin_20" not in first_final_features
assert "PitStop" in first_final_features
assert "PitStop_cat_" in first_final_features
assert "_PitStop_cat_count" in first_final_features
assert "TE_Race_Compound" in first_final_features



[I 2026-05-21 05:12:17,048] A new study created in memory with name: no-name-e5c0400e-9b5c-4369-964e-2f5cde529c8a



Optuna Search: LGB 040 shared001v2 features
Baseline 040 CV: 0.9524028165133429
Optuna seeds: [3407]
N_TRIALS: 40
TIMEOUT_SEC: 36000
final feature count: 167
final features: ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'W_AirTemp_mean', 'W_AirTemp_min', 'W_AirTemp_max', 'W_AirTemp_std', 'W_TrackTemp_mean', 'W_TrackTemp_min', 'W_TrackTemp_max', 'W_TrackTemp_std', 'W_Humidity_mean', 'W_Humidity_min', 'W_Humidity_max', 'W_Humidity_std', 'W_Pressure_mean', 'W_Pressure_std', 'W_WindSpeed_mean', 'W_WindSpeed_max', 'W_WindSpeed_std', 'W_Rainfall_mean', 'W_Rainfall_max', 'W_Rainfall_sum', 'W_Rainfall_count', 'W_TrackTemp_minus_AirTemp_mean', 'W_TrackTemp_minus_AirTemp_max', 'W_AirTemp_range', 'W_TrackTemp_range', 'W_Rainfall_any', 'W_Rainfall_rate', 'W_WetRace', 'WeatherMissing', 'EstimatedTotalLaps', 'LapsRemaining', 'TyreAgeRatio', 'DegPerTyreLap', 

[I 2026-05-21 05:42:56,840] Trial 0 finished with value: 0.951612227778031 and parameters: {'learning_rate': 0.017804685790706463, 'num_leaves': 127, 'min_child_samples': 192, 'subsample': 0.9414456998419366, 'colsample_bytree': 0.6720672934065315, 'reg_alpha': 4.933016817884916, 'reg_lambda': 36.65082446385629, 'max_bin': 127, 'max_depth': 9}. Best is trial 0 with value: 0.951612227778031.


trial 0: cv=0.9516122278, params={'learning_rate': 0.017804685790706463, 'num_leaves': 127, 'min_child_samples': 192, 'subsample': 0.9414456998419366, 'subsample_freq': 1, 'colsample_bytree': 0.6720672934065315, 'reg_alpha': 4.933016817884916, 'reg_lambda': 36.65082446385629, 'max_bin': 127, 'max_depth': 9}


[I 2026-05-21 05:53:10,997] Trial 1 finished with value: 0.9516430310399601 and parameters: {'learning_rate': 0.01455159688167618, 'num_leaves': 15, 'min_child_samples': 146, 'subsample': 0.81179673266914, 'colsample_bytree': 0.8580846811516638, 'reg_alpha': 0.08730190770396953, 'reg_lambda': 22.50425054827572, 'max_bin': 255, 'max_depth': 7}. Best is trial 1 with value: 0.9516430310399601.


...

trial 25: cv=0.9519349544, params={'learning_rate': 0.016541909696642453, 'num_leaves': 127, 'min_child_samples': 104, 'subsample': 0.740351150634882, 'subsample_freq': 1, 'colsample_bytree': 0.8176999611337538, 'reg_alpha': 0.2001481852094869, 'reg_lambda': 33.434565401354604, 'max_bin': 255, 'max_depth': 7}


[I 2026-05-21 12:00:19,663] Trial 26 finished with value: 0.9520568945266749 and parameters: {'learning_rate': 0.02480372048046276, 'num_leaves': 31, 'min_child_samples': 115, 'subsample': 0.7344439254917127, 'colsample_bytree': 0.8525086443756581, 'reg_alpha': 2.916239140851436, 'reg_lambda': 10.330400392653443, 'max_bin': 255, 'max_depth': 7}. Best is trial 19 with value: 0.9523591692420126.


trial 26: cv=0.9520568945, params={'learning_rate': 0.02480372048046276, 'num_leaves': 31, 'min_child_samples': 115, 'subsample': 0.7344439254917127, 'subsample_freq': 1, 'colsample_bytree': 0.8525086443756581, 'reg_alpha': 2.916239140851436, 'reg_lambda': 10.330400392653443, 'max_bin': 255, 'max_depth': 7}


[I 2026-05-21 12:24:38,151] Trial 27 finished with value: 0.9515420291650593 and parameters: {'learning_rate': 0.020089395901510872, 'num_leaves': 95, 'min_child_samples': 59, 'subsample': 0.7659467311580613, 'colsample_bytree': 0.8899852798247672, 'reg_alpha': 0.26839739217288616, 'reg_lambda': 32.61289052724116, 'max_bin': 255, 'max_depth': 7}. Best is trial 19 with value: 0.9523591692420126.


trial 27: cv=0.9515420292, params={'learning_rate': 0.020089395901510872, 'num_leaves': 95, 'min_child_samples': 59, 'subsample': 0.7659467311580613, 'subsample_freq': 1, 'colsample_bytree': 0.8899852798247672, 'reg_alpha': 0.26839739217288616, 'reg_lambda': 32.61289052724116, 'max_bin': 255, 'max_depth': 7}


In [ ]:
# Outputs.
trials_df = study.trials_dataframe()
trials_df.to_csv(CFG.OUTDIR / "optuna_trials_047.csv", index=False)

with open(CFG.OUTDIR / "optuna_study_047.pkl", "wb") as f:
    pickle.dump(study, f)

best_payload = {
    "experiment_id": CFG.EXP_ID,
    "source_experiment": "exp_20260519_040_lgb_shared001v2_features_min",
    "best_trial_number": int(best_trial.number),
    "best_cv_auc": best_value,
    "best_params": best_params,
    "baseline_040_cv_auc": CFG.BASELINE_040_CV,
    "delta_vs_040_cv": best_value - CFG.BASELINE_040_CV,
    "search_uses_seeds": CFG.OPTUNA_SEEDS,
    "refit_should_use_seeds": CFG.SEEDS_BASE,
    "fixed_params": {
        "n_estimators": CFG.LGB_N_ESTIMATORS,
        "early_stopping_rounds": CFG.LGB_EARLY_STOPPING,
        "class_weight": "balanced",
        "device_requested": CFG.LGB_DEVICE,
    },
}
with open(CFG.OUTDIR / "best_params_047.json", "w", encoding="utf-8") as f:
    json.dump(best_payload, f, ensure_ascii=False, indent=2)

study_summary = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-21",
        "type": "optuna_search",
    },
    "base": {
        "experiment": "exp_20260519_040_lgb_shared001v2_features_min",
        "baseline_cv_auc": CFG.BASELINE_040_CV,
        "baseline_public_lb": CFG.BASELINE_040_PUBLIC_LB,
    },
    "optuna": {
        "direction": "maximize",
        "n_trials_requested": CFG.OPTUNA_N_TRIALS,
        "n_trials_completed": len(study.trials),
        "timeout_sec": CFG.OPTUNA_TIMEOUT_SEC,
        "sampler": "TPESampler",
        "seed": CFG.SEED,
        "search_seeds": CFG.OPTUNA_SEEDS,
    },
    "best": {
        "trial_number": int(best_trial.number),
        "cv_auc": best_value,
        "params": best_params,
        "delta_vs_040_cv": best_value - CFG.BASELINE_040_CV,
        "fold_scores": best_trial.user_attrs.get("fold_scores"),
        "fold_mean": best_trial.user_attrs.get("fold_mean"),
        "fold_std": best_trial.user_attrs.get("fold_std"),
        "best_iterations": best_trial.user_attrs.get("best_iterations"),
        "used_devices": best_trial.user_attrs.get("used_devices"),
    },
    "artifacts": {
        "best_params": str(CFG.OUTDIR / "best_params_047.json"),
        "trials": str(CFG.OUTDIR / "optuna_trials_047.csv"),
        "study_pickle": str(CFG.OUTDIR / "optuna_study_047.pkl"),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "study_summary": str(CFG.OUTDIR / "optuna_study_summary_047.json"),
    },
}
with open(CFG.OUTDIR / "optuna_study_summary_047.json", "w", encoding="utf-8") as f:
    json.dump(study_summary, f, ensure_ascii=False, indent=2)

memo_yml = f"""experiment:
  id: exp_20260521_047_lgb_shared001v2_features_optuna_search_min
  title: LGB 040 shared001v2 features Optuna search
  competition: PS S6E5 Predicting F1 Pit Stops
  metric: AUC
  created_at: 2026-05-21
  type: optuna_search
  status: completed

objective:
  summary: >
    040 LGB shared001v2 features構成を固定し、LightGBMの主要ハイパラのみをOptunaで探索する。
    特徴量追加やstack変更は行わず、040単体CVを改善できるか確認する。
  base_experiment: exp_20260519_040_lgb_shared001v2_features_min
  next_step: exp_20260521_048_lgb_shared001v2_features_optuna_refit_min

fixed:
  feature_set: same_as_040
  folds: same_as_040
  original_rows: same_as_040
  target: PitNextLap
  metric: AUC
  forbidden:
    - Normalized_TyreLife
    - Normalized_TyreLife_proxy
    - future_endpoint_proxy
    - pseudo_label
    - new_feature_engineering
    - 039_LOG_features

search_space:
  learning_rate: [0.010, 0.050]
  num_leaves: [15, 31, 63, 95, 127]
  min_child_samples: [40, 260]
  subsample: [0.70, 0.95]
  colsample_bytree: [0.65, 0.95]
  reg_alpha: [0.001, 5.0]
  reg_lambda: [1.0, 50.0]
  max_bin: [127, 255]
  max_depth: [-1, 5, 7, 9]

optuna:
  direction: maximize
  n_trials_requested: {CFG.OPTUNA_N_TRIALS}
  n_trials_completed: {len(study.trials)}
  timeout_sec: {CFG.OPTUNA_TIMEOUT_SEC}
  sampler: TPESampler
  seed: {CFG.SEED}
  search_seeds: {CFG.OPTUNA_SEEDS}
  note: >
    探索段階は計算量を抑えるため1 seedで実施。
    048 refitでは040と同じ2 seed ensembleに戻す。

baseline:
  cv_040: {CFG.BASELINE_040_CV:.12f}
  public_lb_040: {CFG.BASELINE_040_PUBLIC_LB:.5f}

result:
  best_trial_number: {int(best_trial.number)}
  best_cv_auc: {best_value:.12f}
  delta_vs_040_cv: {best_value - CFG.BASELINE_040_CV:+.12f}
  best_params: {json.dumps(best_params, ensure_ascii=False)}

artifacts:
  best_params: best_params_047.json
  trials: optuna_trials_047.csv
  study_summary: optuna_study_summary_047.json
  study_pickle: optuna_study_047.pkl
  feature_columns: feature_columns.csv

judgement:
  decision: pending_refit
  reason: >
    047は探索実験であり、正式OOF/pred/submissionは作成しない。
    best paramsを048へ適用し、040と同じ2 seed ensembleで正式refitしてから採否判断する。
"""
with open(CFG.OUTDIR / "memo_draft.yml", "w", encoding="utf-8") as f:
    f.write(memo_yml)

print("Saved to:", CFG.OUTDIR)
print("best_params:", CFG.OUTDIR / "best_params_047.json")
print("best_cv_auc:", best_value)
